# 42. Retrieval-augmented generation and evidence

**Level:** Intermediate to advanced

**Prerequisites:** 33-34 and 40: vector text features and prompt boundaries.

[Course index](00_START_HERE.ipynb) | [Previous](41_Structured_Output_and_Tool_Calling.ipynb) | [Next](43_LLM_Evaluation_Hallucinations_and_Reliability.ipynb)

**Topics:** RAG, Retrieval, Chunking, Vector search, Citations, Abstention.

**Matching script:** `examples\42_Retrieval_Augmented_Generation.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

Retrieval-augmented generation retrieves external evidence and supplies it to a generator.
Retrieval can use lexical search, dense embeddings, hybrid search, and reranking. It does not
retrain model weights on every query. A retriever can fail before generation starts.

Document chunking defines the evidence units. Tiny chunks can lose context; huge chunks waste
budget and mix unrelated claims. Store stable source IDs, offsets, version dates, access
controls, and provenance. Retrieval must enforce document permissions before disclosure.

A generator should ground claims in relevant evidence and abstain when support is missing.
A citation that merely names a document is not proof the document supports the specific claim.
Measure retrieval recall and grounded answer quality separately.

This lesson implements retrieval plus deterministic extractive answering using TF-IDF. It is
explicitly the retrieval/evidence half of a RAG architecture, not a disguised pretrained LLM.
A real generator can later consume the same evidence contract after review and evaluation.

## Visual guide

![Retrieval-augmented generation and evidence teaching diagram](images/rag.png)

Trace the diagram left to right and identify the input, transformation, and output. It illustrates the mechanism, not measured model quality.

## Mathematics or data contract

For L2-normalized TF-IDF rows, dot(query_vector, document_vector) equals cosine similarity.

## Important limitation

Lexical overlap is not semantic entailment; a fixed retrieval threshold needs validation.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Retrieve with source IDs and abstain on no overlap

**Purpose:** Keep evidence provenance visible.

### Read the code in this order
1. Fit a local document index.
2. Score a query using normalized sparse vectors.
3. Return a source-backed extract rather than inventing a generated answer.
4. Assert a relevant source and an unsupported-query abstention.

### Syntax and behavior to notice
- `transform(...)` applies already-fitted preprocessing; it does not refit it.
- `assert` checks the stated behavior; a failed check raises AssertionError.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
{'answer': 'The lab interface MTU is 1500 bytes.', 'sources': ['mtu-policy']}
{'answer': 'unknown', 'sources': []}
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

documents = [
    {"id": "mtu-policy", "text": "The lab interface MTU is 1500 bytes."},
    {"id": "dns-policy", "text": "The lab DNS resolver is 192.0.2.53."},
    {"id": "backup-policy", "text": "Configuration backups run every day."},
]
vectorizer = TfidfVectorizer()
index = vectorizer.fit_transform([item["text"] for item in documents])
def retrieve(query):
    scores = (index @ vectorizer.transform([query]).T).toarray().ravel()
    best = int(np.argmax(scores))
    if scores[best] == 0:
        return {"answer": "unknown", "sources": []}
    document = documents[best]
    return {"answer": document["text"], "sources": [document["id"]]}
result = retrieve("interface MTU bytes")
assert result["sources"] == ["mtu-policy"] and "1500" in result["answer"]
assert retrieve("penguins Antarctica") == {"answer": "unknown", "sources": []}
print(result)
print(retrieve("penguins Antarctica"))

{'answer': 'The lab interface MTU is 1500 bytes.', 'sources': ['mtu-policy']}
{'answer': 'unknown', 'sources': []}


### Interpret example 1

The result should demonstrate this purpose: **Keep evidence provenance visible.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Add documents with similar words but contradictory facts and design an ambiguity/abstention test.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `RAG` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: For L2-normalized TF-IDF rows, dot(query_vector, document_vector) equals cosine similarity.
- Name the boundary of the demonstration: Lexical overlap is not semantic entailment; a fixed retrieval threshold needs validation.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://scikit-learn.org/stable/user_guide.html`
- `https://arxiv.org/abs/2005.11401`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.